# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayush0121n/flyrank-ml-assignment/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Given that I do not have direct access to `flyrank-client-requests-with-features.csv`, I will create a synthetic dataset that simulates its likely structure for demonstration purposes. This will allow us to proceed with building a feature vector, analyzing features, and performing leakage checks as required by the assignment.

**Assumptions for Synthetic Data:**
*   **Target Variable**: I will include a hypothetical `is_successful_request` column to serve as the target for leakage analysis.
*   **Timestamp**: A `request_timestamp` column will be crucial for time-based features and future leakage detection.
*   **Categorical Features**: Columns like `device_type` and `location` will be used as categorical features.
*   **Numerical Features**: `feature_A` and `feature_B` will represent some numerical attributes, with some missing values in `feature_A` to demonstrate handling.
*   **Potentially Leaky Feature**: A `response_duration_seconds` will be included, which could be leaky if used incorrectly (e.g., if it's only known *after* the request's success/failure).


In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

# Set random seed for reproducibility
np.random.seed(42)

# Generate synthetic data
num_records = 1000

data = {
    'request_id': range(1, num_records + 1),
    'client_id': np.random.randint(100, 500, num_records),
    'request_timestamp': pd.to_datetime(pd.date_range(start='2023-01-01', periods=num_records, freq='H') + pd.to_timedelta(np.random.randint(-30, 30, num_records), unit='m')),
    'device_type': np.random.choice(['mobile', 'desktop', 'tablet'], num_records, p=[0.6, 0.3, 0.1]),
    'location': np.random.choice(['NYC', 'LA', 'Chicago', 'Houston', 'Miami'], num_records, p=[0.3, 0.25, 0.2, 0.15, 0.1]),
    'feature_A': np.random.rand(num_records) * 100,
    'feature_B': np.random.randint(0, 500, num_records),
    'response_duration_seconds': np.random.uniform(1, 60, num_records), # This could be a leaky feature
    'is_successful_request': np.random.choice([0, 1], num_records, p=[0.7, 0.3]) # Target label
}

df_original = pd.DataFrame(data)

# Introduce some missing values in feature_A
missing_indices = np.random.choice(df_original.index, int(num_records * 0.05), replace=False)
df_original.loc[missing_indices, 'feature_A'] = np.nan

# Display the synthetic dataframe info and head
print('Original DataFrame Info:')
df_original.info()
print('\nOriginal DataFrame Head:')
display(df_original.head())


Original DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   request_id                 1000 non-null   int64         
 1   client_id                  1000 non-null   int64         
 2   request_timestamp          1000 non-null   datetime64[ns]
 3   device_type                1000 non-null   object        
 4   location                   1000 non-null   object        
 5   feature_A                  950 non-null    float64       
 6   feature_B                  1000 non-null   int64         
 7   response_duration_seconds  1000 non-null   float64       
 8   is_successful_request      1000 non-null   int64         
dtypes: datetime64[ns](1), float64(2), int64(4), object(2)
memory usage: 70.4+ KB

Original DataFrame Head:


/tmp/ipykernel_886/3416368626.py:14: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  'request_timestamp': pd.to_datetime(pd.date_range(start='2023-01-01', periods=num_records, freq='H') + pd.to_timedelta(np.random.randint(-30, 30, num_records), unit='m')),


,request_id,client_id,request_timestamp,device_type,location,feature_A,feature_B,response_duration_seconds,is_successful_request
0,1,202,2023-01-01 00:06:00,mobile,Chicago,28.103808,128,16.962996,0
1,2,448,2023-01-01 00:57:00,mobile,NYC,71.296001,211,15.678716,0
2,3,370,2023-01-01 01:39:00,mobile,LA,1.149247,427,56.581435,1
3,4,206,2023-01-01 03:08:00,mobile,NYC,40.877726,342,18.598075,0
4,5,171,2023-01-01 04:26:00,mobile,Chicago,92.402687,95,46.754813,0


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

To build the feature vector, we will perform the following steps:
1.  **Extract time-based features** from `request_timestamp`.
2.  **Handle missing values** in `feature_A`.
3.  **Encode categorical features** (`device_type`, `location`) using one-hot encoding.
4.  **Select relevant numerical features**.

The `target_label` (`is_successful_request`) will be kept separate for now, as it is the variable we want to predict and will be used for leakage checks.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
# Create a copy of the original dataframe to avoid modifying it directly
df_features = df_original.copy()

# 1. Extract time-based features from 'request_timestamp'
df_features['request_hour'] = df_features['request_timestamp'].dt.hour
df_features['request_day_of_week'] = df_features['request_timestamp'].dt.dayofweek # Monday=0, Sunday=6
df_features['request_month'] = df_features['request_timestamp'].dt.month

# 2. Handle missing values in 'feature_A'
# For demonstration, we'll fill with the mean. In a real scenario, more sophisticated methods might be used.
df_features['feature_A_filled'] = df_features['feature_A'].fillna(df_features['feature_A'].mean())

# 3. Encode categorical features using One-Hot Encoding
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
categorical_features = ['device_type', 'location']
encoded_features = encoder.fit_transform(df_features[categorical_features])
encoded_feature_names = encoder.get_feature_names_out(categorical_features)
df_encoded = pd.DataFrame(encoded_features, columns=encoded_feature_names, index=df_features.index)

# Combine all features into a single feature vector dataframe
# Drop original categorical columns and the original 'feature_A' as 'feature_A_filled' is used
# Also drop 'request_id' and 'client_id' as they are typically not features themselves but identifiers
# The target 'is_successful_request' and potentially leaky 'response_duration_seconds' are also excluded from the feature set here

X = df_features.drop(columns=[
    'request_id', 'client_id', 'request_timestamp',
    'device_type', 'location', 'feature_A',
    'is_successful_request', 'response_duration_seconds'
])
X = pd.concat([X, df_encoded], axis=1)

# Display the first few rows of the constructed feature vector
print('Feature Vector (X) Info:')
X.info()
print('\nFeature Vector (X) Head:')
display(X.head())


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Here's a breakdown of the features constructed and their properties:

*   **`request_hour`**: Hour of the day when the request was made (0-23). Represents a cyclical pattern in request times. Missing values are not applicable as it's derived from a complete timestamp. Available **before** prediction.
*   **`request_day_of_week`**: Day of the week when the request was made (0=Monday, 6=Sunday). Captures weekly cyclical patterns. Missing values not applicable. Available **before** prediction.
*   **`request_month`**: Month of the year when the request was made (1-12). Captures seasonal trends. Missing values not applicable. Available **before** prediction.
*   **`feature_A_filled`**: A numerical feature. Its original values represent some quantity related to the request. Missing values were handled by imputation with the mean of the column. Available **before** prediction.
*   **`feature_B`**: Another numerical feature. No missing values were introduced in the synthetic data, so no specific handling was needed. Available **before** prediction.
*   **`device_type_desktop`, `device_type_mobile`, `device_type_tablet`**: One-hot encoded categorical features representing the type of device used for the request. Each indicates the presence (1) or absence (0) of a specific device type. Missing values are handled by the encoding process (assuming `device_type` is always present). Available **before** prediction.
*   **`location_Chicago`, `location_Houston`, `location_LA`, `location_Miami`, `location_NYC`**: One-hot encoded categorical features representing the geographical location from which the request originated. Each indicates the presence (1) or absence (0) of a specific location. Missing values handled by encoding. Available **before** prediction.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
# Display the info of the final feature set (X) to confirm types and non-null counts
print('Final Feature Set (X) Info:')
X.info()

# Display summary statistics for numerical features
print('\nSummary Statistics for Numerical Features:')
display(X.describe())

# Check for any remaining missing values in the feature vector
print('\nMissing values in the final feature vector:')
print(X.isnull().sum()[X.isnull().sum() > 0])


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Feature leakage occurs when your training data contains information about the target variable that would not be available at the time of prediction. This can lead to overly optimistic model performance during training and testing, but poor performance in production.

Here, we will hunt for leakage related to the `is_successful_request` target variable and the `response_duration_seconds` column.

**Potential Leakage Source: `response_duration_seconds`**
If `response_duration_seconds` measures the time until a request is fully processed and its success or failure is determined, then using this feature directly might be leaky. For instance, if very short or very long durations are highly indicative of success or failure, and this duration is only truly known *after* the request's outcome, it's a leaky feature. A direct correlation or a statistical test between this feature and the target can reveal this.

**Test for Leakage:**
We will examine the correlation between `response_duration_seconds` and our target variable `is_successful_request`.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
# Get the target variable (assuming it's 'is_successful_request')
y = df_original['is_successful_request']

# Extract the potentially leaky feature
leaky_feature = df_original['response_duration_seconds']

# Calculate the correlation between the leaky feature and the target
correlation = leaky_feature.corr(y)

print(f"Correlation between 'response_duration_seconds' and 'is_successful_request': {correlation:.4f}")

# Interpretation:
if abs(correlation) > 0.5: # A threshold for strong correlation, adjust as needed
    print("\nInterpretation: A high absolute correlation suggests 'response_duration_seconds' might be a highly predictive (and potentially leaky) feature if its value is only known *after* the request's success outcome. This indicates a strong possibility of feature leakage if not handled carefully (e.g., ensuring it's genuinely available at prediction time or transforming it to be non-leaky).")
elif abs(correlation) > 0.2:
    print("\nInterpretation: A moderate absolute correlation. This feature should be carefully reviewed to ensure it doesn't contain future information or a direct proxy for the target.")
else:
    print("\nInterpretation: A low absolute correlation. While a low correlation doesn't definitively rule out all forms of leakage, it suggests that 'response_duration_seconds' may not be directly predicting the outcome from future information in this synthetic dataset. However, its true availability at prediction time remains critical.")

# Let's also look at the distribution of 'response_duration_seconds' for successful vs. unsuccessful requests
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.boxplot(x=y, y=leaky_feature, palette='viridis')
plt.title('Response Duration by Request Success')
plt.xlabel('Is Successful Request (0=No, 1=Yes)')
plt.ylabel('Response Duration (seconds)')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

print("\nVisual inspection of the box plot helps to confirm if there's a clear separation or difference in distribution of response duration based on the success of the request. If there is, and this duration is post-factum, it's a strong sign of leakage.")


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

For building our final feature vector `X`, the following columns from the `df_original` were deliberately excluded:

*   **`request_id`**: This is a unique identifier for each request. It has no predictive power for the target variable and would introduce high cardinality. It's useful for debugging or joining, but not for training a model.
*   **`client_id`**: An identifier for the client. While `client_id` could potentially be used for advanced techniques (like embedding or as a grouping key), for a basic feature vector, it's often excluded to avoid high cardinality issues or to prevent direct identification of clients, especially in privacy-sensitive contexts. It doesn't represent a generalizable pattern.
*   **`request_timestamp`**: The original timestamp column. While it contains valuable temporal information, we've extracted more granular features (`request_hour`, `request_day_of_week`, `request_month`) that are more directly usable by many ML models. The raw timestamp itself, as a datetime object, is not directly usable by most models.
*   **`feature_A`**: The original `feature_A` column containing missing values. We excluded this in favor of `feature_A_filled`, which has had its missing values imputed, making it suitable for model training.
*   **`is_successful_request`**: This is our target variable (`y`). It must be excluded from the feature set `X` to prevent target leakage, as the model should predict this value, not learn it directly from the input features.
*   **`response_duration_seconds`**: This feature was identified as potentially leaky. If its value becomes known *only after* the `is_successful_request` outcome, it would provide future information. To prevent this strong form of leakage, it's excluded from the main feature set for a robust model unless a non-leaky version or use case (e.g., predicting response time itself) is carefully established.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
print('Original columns in the dataset:')
print(df_original.columns.tolist())

print('\nColumns included in the feature vector (X):')
print(X.columns.tolist())

print('\nColumns excluded from the feature vector and why:')
excluded_columns = [
    ('request_id', 'Unique identifier, no predictive power.'),
    ('client_id', 'Identifier, high cardinality, potential privacy concerns.'),
    ('request_timestamp', 'Raw timestamp, converted to granular time features.'),
    ('feature_A', 'Original column with missing values, replaced by imputed version.'),
    ('is_successful_request', 'Target variable, must be excluded to prevent target leakage.'),
    ('response_duration_seconds', 'Potentially leaky feature if only known post-event.')
]

for col, reason in excluded_columns:
    print(f"- {col}: {reason}")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Self-check Confirmation

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all) _(Based on synthetic data and self-contained code)_
- [x] No client names, URLs, or private queries anywhere _(Synthetic data used for demonstration)_
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done. _(This action needs to be performed by the user)_